## Process Notes
I am thinking about the roadmap and tasks for this project

1: Data collection and EDA

- data size, schema, file format
- useful column identification
- inspect cleanliness, like missing values/duplicates/weird dates

2: Data cleaning

- check if need to standardize
- make sure dates are in datetime
- handle missing values, duplicates, etc

3: Parallel Processing

- find computationally expensive operations
- cleaning, categorizing, aggregating could be expensive
- processing time comparison for paper?

4: Distributed Computing with Spark

- make the spark cluster
- connect to S3 bucket and read in CSV
- transformations and aggregations: this is where the actual computation happens
- demographic distributions, program-type statistics, monthly intakes, and immediate-needs frequencies.
- write processed results back to S3, maybe in Parquet format

5: Analysis
- analyze all the calculations we did in the Spark cluster
- NLP of intake_note
- monthly aggregations

6: Dashboard
- build all the charts and interface in a Streamlit dashboard


# Spark Cluster Setup
This section will access the S3 bucket, extract its schema, and evaluate a little more about the dataset contents. This reiquired connecting to worker nodes via Apache Spark

In [1]:
import polars as pl

# creating a lazy frame with a polars scan of the csv
nyc_lf = pl.scan_csv('s3://dsan6000-final/nyc-shelters/intake_log.csv')
nyc_schema = nyc_lf.collect_schema()

print(nyc_schema)

# this all checks out with the codebook. This shows the column names and data types

<positron-console-cell-1>:5: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.


Schema([('intake_id', String), ('synthetic_data', Boolean), ('client_display_name', String), ('age', Int64), ('household_type', String), ('children_count', Int64), ('program_type', String), ('contact_type', String), ('intake_date', String), ('presenting_reason', String), ('prior_living_situation_reported', String), ('reported_housing_instability_days', Int64), ('employment_status_reported', String), ('income_status_reported', String), ('primary_barrier', String), ('immediate_needs', String), ('potential_referral', String), ('referral_status', String), ('client_goal', String), ('follow_up_days', Int64), ('follow_up_target_date', String), ('housing_eligibility_verified', Boolean), ('staff_role', String), ('note_word_count', Int64), ('intake_note', String), ('intake_photo_b64', String)])


In [2]:
# overview: How many interactions are there each day in this dataset
daily_lf = (
    nyc_lf
    .group_by("intake_date")
    .agg(pl.len().alias("intake_id"))
    .sort("intake_date")
)


# Materialize the plan into yearly_pl, and display its contents
daily_pl = daily_lf.collect()

daily_pl

intake_date,intake_id
str,u32
"""2026-02-14""",449
"""2026-02-15""",409
"""2026-02-16""",401
"""2026-02-17""",460
"""2026-02-18""",439
…,…
"""2026-11-07""",404
"""2026-11-08""",397
"""2026-11-09""",428


So, this data goes from February 14 2026 to November 11th 2026. There are roughly 400 intakes per day, over 270 days. There are roughly 108,000 entries, based on that calculation.

Upon trying to count the number of rows in this dataset, I hit the storage limit. We will move this EDA into Spark

In [ ]:
# the following code cells are copied from hw 07
import pyspark
from pyspark.sql import SparkSession

prev_spark = SparkSession.getActiveSession()
if prev_spark is not None:
    prev_spark.stop()
else:
    print("No previously-running SparkSessions found")

No previously-running SparkSessions found


In [6]:
#| label: get-private-ip
# Get the private IP address for this instance
hostname_output = !hostname -I
master_private_ip = hostname_output[0].strip()
print(f"Master node private IP address: {master_private_ip}")

Master node private IP address: 172.31.15.53


In [7]:
spark = (
    SparkSession
      .builder
      .appName('NYC_Shelters')
      .master(f'spark://{master_private_ip}:7077')
      .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
      .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/home/ubuntu/classwork/dsan_6000_final/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8e869d6-e6e9-4198-a040-dc488d271722;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 617ms :: artifacts dl 29ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 fro

In [8]:
print(spark.sparkContext.master)

spark://172.31.15.53:7077


Here is the code I will have to copy a few times to initialize the workers

sudo apt update

sudo apt install openjdk-21-jdk-headless

wget https://dlcdn.apache.org/spark/spark-4.3.0/spark-4.3.0-bin-hadoop3.tgz

tar -xvzf spark-4.3.0-bin-hadoop3.tgz

cd spark-4.3.0-bin-hadoop3/sbin

./start-worker.sh 172.31.15.53:7077


In [49]:
# Read the CSV from S3
file_path = "s3a://dsan6000-final/nyc-shelters/intake_log.csv"

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("delimiter", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .csv(file_path)
)

In [50]:
print("Columns:", len(df.columns))
print("Partitions:", df.rdd.getNumPartitions())

df.select(
    "intake_id",
    "age",
    "household_type",
    "intake_date",
    "presenting_reason"
).show(10, truncate=False)

Columns: 26
Partitions: 1
+---------+---+--------------------+-----------+-----------------------------------+
|intake_id|age|household_type      |intake_date|presenting_reason                  |
+---------+---+--------------------+-----------+-----------------------------------+
|SYN-0001 |46 |Single adult        |2026-05-19 |Job loss                           |
|SYN-0002 |32 |Family with children|2026-09-13 |Family housing loss                |
|SYN-0003 |41 |Young adult         |2026-07-13 |Family conflict                    |
|SYN-0004 |44 |Single adult        |2026-08-08 |Long-term housing instability      |
|SYN-0005 |27 |Single adult        |2026-06-28 |Domestic violence safety concern   |
|SYN-0006 |44 |Single adult        |2026-06-18 |Medical-related income interruption|
|SYN-0007 |62 |Single adult        |2026-08-13 |Lease nonrenewal                   |
|SYN-0008 |58 |Single adult        |2026-05-16 |Recent relocation                  |
|SYN-0009 |58 |Single adult        |202

In [53]:
print("Number of partitions:", df.rdd.getNumPartitions())
# I would like more than 1 partition

Number of partitions: 1


In [54]:
print("Default parallelism:",
      spark.sparkContext.defaultParallelism)

print("Input partitions:",
      df.rdd.getNumPartitions())

print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

print("Max input partition bytes:",
      spark.conf.get("spark.sql.files.maxPartitionBytes"))

Default parallelism: 6
Input partitions: 1
Shuffle partitions: 200
Max input partition bytes: 134217728b


In [ ]:
print("Original:", df.rdd.getNumPartitions())

df_test = df.repartition(12)

print("Repartitioned:", df_test.rdd.getNumPartitions())

Original: 1


26/11/24 22:48:40 ERROR TaskSchedulerImpl: Lost executor 0 on 172.31.9.206: Command exited with code 52
26/11/24 22:48:40 WARN TaskSetManager: Lost task 0.0 in stage 49.0 (TID 335) (172.31.9.206 executor 0): ExecutorLostFailure (executor 0 exited caused by one of the running tasks) Reason: Command exited with code 52
26/11/24 22:48:45 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/files/software.amazon.awssdk_bundle-2.35.4.jar,byteCount=686119507,body=FileSegmentManagedBuffer[file=/home/ubuntu/.ivy2.5.2/jars/software.amazon.awssdk_bundle-2.35.4.jar,offset=0,length=686119507]] to /172.31.9.206:47534; closing connection
io.netty.channel.unix.Errors$NativeIoException: sendfile(..) failed with error(-104): Connection reset by peer
26/11/24 22:48:45 ERROR TaskSchedulerImpl: Lost executor 4 on 172.31.9.206: Unable to create executor due to Disk quota exceeded
26/11/24 22:48:50 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/files/sof

## Exploratory Data Analysis

### Data quality and structure

In [45]:
#rows
row_count = df.count()

#columns
column_count = len(df.columns)

print("Number of rows:", row_count)
print("Number of columns:", column_count)

Number of rows: 445000
Number of columns: 26


In [46]:
#checking that Spark has the schema
print(df.columns)
df.printSchema()

['intake_id', 'synthetic_data', 'client_display_name', 'age', 'household_type', 'children_count', 'program_type', 'contact_type', 'intake_date', 'presenting_reason', 'prior_living_situation_reported', 'reported_housing_instability_days', 'employment_status_reported', 'income_status_reported', 'primary_barrier', 'immediate_needs', 'potential_referral', 'referral_status', 'client_goal', 'follow_up_days', 'follow_up_target_date', 'housing_eligibility_verified', 'staff_role', 'note_word_count', 'intake_note', 'intake_photo_b64']
root
 |-- intake_id: string (nullable = true)
 |-- synthetic_data: string (nullable = true)
 |-- client_display_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- household_type: string (nullable = true)
 |-- children_count: string (nullable = true)
 |-- program_type: string (nullable = true)
 |-- contact_type: string (nullable = true)
 |-- intake_date: string (nullable = true)
 |-- presenting_reason: string (nullable = true)
 |-- prior_living_s

In [47]:
#date range
from pyspark.sql import functions as F
date_df = df.withColumn(
    "parsed_intake_date",
    F.expr("try_to_date(trim(intake_date), 'yyyy-MM-dd')")
)

In [48]:
date_summary = date_df.agg(
    F.min("parsed_intake_date").alias("earliest_intake_date"),
    F.max("parsed_intake_date").alias("latest_intake_date"),
    F.count("parsed_intake_date").alias("valid_dates"),
    F.sum(
        F.when(F.col("parsed_intake_date").isNull(), 1).otherwise(0)
    ).alias("invalid_or_missing_dates")
)

date_summary.show(truncate=False)

+--------------------+------------------+-----------+------------------------+
|earliest_intake_date|latest_intake_date|valid_dates|invalid_or_missing_dates|
+--------------------+------------------+-----------+------------------------+
|2026-02-14          |2026-11-11        |111000     |334000                  |
+--------------------+------------------+-----------+------------------------+



There are 334,000 invalid or missing intake dates.

In [ ]:
#duplicates check

In [ ]:
# missing values check

In [ ]:
#invalid dates or ages

In [ ]:
# how many synthetic

In [ ]:
# Ending spark cluster
spark.stop()
# stop all instances in EC2 webpage